In [2]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [1]:
!nvidia-smi

Fri Jan 30 16:07:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from pathlib import Path
from huggingface_hub import snapshot_download

data_id = "alxxtexxr/VRBI-Dataset"
allow_dir = 'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2'

data_dir = snapshot_download(
    repo_id=data_id,
    repo_type='dataset',
    allow_patterns=f'{allow_dir}/**',
    local_dir='data',
    local_dir_use_symlinks=False,
)
vision_data_dir = Path(data_dir) / allow_dir

print("Vision data downloaded to:", vision_data_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

...

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/742M [00:00<?, ?B/s]

model_unsloth_Qwen2.5-VL-3B-Instruct.dat(…):   0%|          | 0.00/49.8k [00:00<?, ?B/s]

Vision data downloaded to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2


In [3]:
import joblib
import numpy as np
from sklearn.cluster import MiniBatchKMeans
import time

# Load all batch file paths and the scaler
batch_paths = [f for f in vision_data_dir.glob('*.npz')]
scaler_path = vision_data_dir / 'scaler.pkl'
scaler = joblib.load(scaler_path)

# Initialize MiniBatchKMeans
k = 16384 # Number of clusters
kmeans = MiniBatchKMeans(
    n_clusters=k,
    init='random',
    batch_size=16384,
    n_init=1,
    reassignment_ratio=0.01,
    max_no_improvement=10,
    random_state=42
)

# Load each batch file one by one and update centroids
# batch_paths -> current: 10 files
for i, batch_path in enumerate(batch_paths):
    print("="*128)
    print(f"Processing batch {i+1}/{len(batch_paths)}: {batch_path}")
    print("="*128)
    
    start_time = time.time()
    
    batch = np.load(batch_path)
    visual_embs = batch['visual_embs'].astype(np.float32) # (N, 2048) -> current: (32_400, 2048)
    visual_embs_norm = scaler.transform(visual_embs) # Normalize using the loaded scaler
    
    print("Normalized visual embeddings shape:", visual_embs_norm.shape)
    print("Mean after normalization (should be close to 0):", visual_embs_norm.mean(axis=0)[:5])
    print("Std after normalization (should be close to 1):", visual_embs_norm.std(axis=0)[:5])
    print()
    
    kmeans.partial_fit(visual_embs_norm)
    
    end_time = time.time()
    iter_time = end_time - start_time
    
    print(f"Time taken: {iter_time:.2f}s")
    print()

Processing batch 1/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/0-99.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean after normalization (should be close to 0): [-0.07018206  0.01279403  0.13627605 -0.07857896 -0.2096578 ]
Std after normalization (should be close to 1): [1.0386101 1.0893176 1.0412676 1.0409397 1.0361084]

Time taken: 51.91s

Processing batch 2/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/300-399.vision_data.npz
Normalized visual embeddings shape: (32400, 2048)
Mean after normalization (should be close to 0): [ 0.0138678  -0.010088   -0.07950611  0.09919769  0.10777523]
Std after normalization (should be close to 1): [0.98345035 0.979839   0.97040653 0.96668404 0.95598763]

Time taken: 50.03s

Processing batch 3/10: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/500-599.vision_data.npz
Normalized visual embeddings shape: (3240

In [6]:
kmeans_path = vision_data_dir / 'kmeans.pkl'
centroids_path = vision_data_dir / 'centroids.npy'

joblib.dump(kmeans, kmeans_path)
np.save(centroids_path, kmeans.cluster_centers_)

print(f"K-means model saved to: {kmeans_path}")
print(f"Centroids saved to: {centroids_path}")

K-means model saved to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/kmeans.pkl
Centroids saved to: /content/data/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/centroids.npy


In [12]:
from huggingface_hub import upload_file

upload_file(
    path_or_fileobj=str(kmeans_path),
    path_in_repo=str(kmeans_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)
upload_file(
    path_or_fileobj=str(centroids_path),
    path_in_repo=str(centroids_path.relative_to('/content/data')),
    repo_id='alxxtexxr/VRBI-Dataset',
    repo_type='dataset',
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tions_train.v2/kmeans.pkl:   0%|          |  552kB /  134MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ns_train.v2/centroids.npy:  31%|###1      | 41.9MB /  134MB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset/commit/e3a884ba3f4fb4818bba22fa6c61d5f4f7d42eb0', commit_message='Upload model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/centroids.npy with huggingface_hub', commit_description='', oid='e3a884ba3f4fb4818bba22fa6c61d5f4f7d42eb0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/VRBI-Dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/VRBI-Dataset'), pr_revision=None, pr_num=None)